In [ ]:

import os 
from typing import List, Dict
from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()


In [ ]:
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not gemini_api_key:
    raise ValueError("Gemini API key not found. Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your .env file.")

print("Gemini API key loaded successfully.")

In [ ]:
class AgentState(TypedDict):
    question: str
    Documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
from langchain import embeddings

sample_text = "This is a sample document for testing the embedding model. It contains multiple sentences to ensure that the text splitter works correctly."

Document = [Document(page_content=sample_text, metadata={"source": "sample.txt"})]

vectorstore = FAISS.from_documents(Document, embeddings)
retriever = vectorstore.as_retriever(k = 3)

In [ ]:
def decide_retrieval(state: AgentState) -> bool:
     question = state["question"]
     
     retrieval_keywords = ["who", "what", "when", "where", "why", "how", "which", "list", "define", "explain"]
     needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)
     return{**state, "needs_retrieval": needs_retrieval}

In [ ]:
def should_retrieve(state: AgentState) -> str:
    return "yes" if state["needs_retrieval"] else "no"

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("decide_retrieval", decide_retrieval)
workflow.add_node("should_retrieve", should_retrieve)
workflow.add_edge("decide_retrieval", "should_retrieve")
workflow.add_edge("should_retrieve", "generate")
workflow.visualize()

workflow.add_conditional_edges(
    "decide",should_retrieve,{
    "yes": "retrieve",
    "no": "generate"

    }
)
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

In [ ]:
question1 = "What is the capital of France?"
initial_state = AgentState(
    question=question1,
    Documents=[],
    answer="",
    needs_retrieval=False
)
result = workflow.run(initial_state)
print(result)

In [ ]:
#Due to the fact that the generate and retrieve nodes are not implemented, we will only run the workflow up to the decide_retrieval node for now.
#api key is not used so the workflow will not run successfully, but this is just to demonstrate the structure of the workflow.
